<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/notebooks/03_classifier_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Classifier training

Trains ResNet18 and EfficientNet-B0 (ImageNet-pretrained, fully fine-tuned) as binary tampered/authentic classifiers on the full validated CASIA v2 dataset (5,123 tampered + 7,491 authentic — both copy-move and spliced images, since the classifier's task is general tamper detection, not the size/polarity study itself).

**These classifiers exist to produce non-degenerate Grad-CAM gradients for notebook 04, not to achieve state-of-the-art detection accuracy.** A reasonably converged, non-collapsed model is the actual requirement — not maximum accuracy.

**If you already have validated checkpoints on Drive (`casia_resnet_best.pt`, `casia_effnet_best.pt`), you do not need to re-run training.** Skip to the "Load existing checkpoints" section at the end to verify they load correctly instead.

## Setup — mount Drive, rebuild validated file lists

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import random
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
from PIL import Image

random.seed(42)
torch.manual_seed(42)

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
au_dir = os.path.join(base, "Au")
tp_dir = os.path.join(base, "Tp")

with open(os.path.join(base, "au_list.txt")) as f:
    au_list_content = [line.strip() for line in f if line.strip()]
with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]

au_files_final = sorted(set(au_list_content) & set(os.listdir(au_dir)))
tp_files_final = sorted(set(tp_list_content) & set(os.listdir(tp_dir)))

print(f"Authentic: {len(au_files_final)} | Tampered: {len(tp_files_final)}")  # expect 7491, 5123

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Mounted at /content/drive
Authentic: 7491 | Tampered: 5123
Device: cuda


## Seeded train/val split
Saved to Drive so it can be reproduced exactly across sessions, rather than regenerated with a different random split each time.

In [2]:
random.shuffle(tp_files_final)
random.shuffle(au_files_final)

def split(files, val_frac=0.2):
    cut = int(len(files) * (1 - val_frac))
    return files[:cut], files[cut:]

train_tp, val_tp = split(tp_files_final)
train_au, val_au = split(au_files_final)

split_dict = {"train_tampered": train_tp, "val_tampered": val_tp,
              "train_authentic": train_au, "val_authentic": val_au, "seed": 42}
with open("/content/drive/MyDrive/CASIA2.0/casia_split.json", "w") as f:
    json.dump(split_dict, f)

print(f"Train: {len(train_tp)+len(train_au)} | Val: {len(val_tp)+len(val_au)}")

Train: 10090 | Val: 2524


## Dataset, transforms, and DataLoaders
Augmentation (flip, rotation, color jitter) applied only to training data — validation data is left unaugmented so it reflects real performance.

In [3]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CASIADataset(Dataset):
    def __init__(self, tampered_files, authentic_files, tampered_dir, authentic_dir, transform):
        self.samples = [(os.path.join(tampered_dir, f), 1) for f in tampered_files] + \
                        [(os.path.join(authentic_dir, f), 0) for f in authentic_files]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_dataset = CASIADataset(train_tp, train_au, tp_dir, au_dir, train_transform)
val_dataset = CASIADataset(val_tp, val_au, tp_dir, au_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 316 | Val batches: 79


## Generic training loop
Shared by both architectures below. Early stopping on validation loss (patience=4) prevents chasing epochs past the point of diminishing returns — a mild train/val divergence after the best epoch is expected and handled by keeping the best checkpoint, not by training longer.

In [ ]:
def train_model(model, model_name, max_epochs=30, patience=4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "val_f1": [], "val_balanced_acc": []}
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds)
        val_bal_acc = balanced_accuracy_score(all_labels, all_preds)

        scheduler.step(val_loss)
        for k, v in zip(history.keys(), [train_loss, val_loss, train_acc, val_acc, val_f1, val_bal_acc]):
            history[k].append(v)

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_bal_acc={val_bal_acc:.4f}")

        torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_last.pt")
        with open(f"/content/drive/MyDrive/CASIA2.0/{model_name}_history.json", "w") as f:
            json.dump(history, f)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_best.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    return model, history

## Train ResNet18
**Optional — skip if `casia_resnet_best.pt` already exists and is validated.**

In [ ]:
resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet, resnet_history = train_model(resnet, "casia_resnet")

## Train EfficientNet-B0
**Optional — skip if `casia_effnet_best.pt` already exists and is validated.**

In [ ]:
effnet = models.efficientnet_b0(weights="IMAGENET1K_V1")
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet, effnet_history = train_model(effnet, "casia_effnet")

## Load existing checkpoints (recommended path if you already have validated models)
Loads the best saved checkpoint for each architecture without retraining.

In [5]:
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()

print("Both checkpoints loaded successfully.")

Both checkpoints loaded successfully.


## Confusion matrix check
Confirms balanced recall across both classes (no majority-class collapse) for whichever model is currently loaded above.

In [6]:
model_to_check = resnet  # swap to effnet to check the other architecture

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_to_check(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Authentic", "Tampered"]))

[[1177  322]
 [ 153  872]]
              precision    recall  f1-score   support

   Authentic       0.88      0.79      0.83      1499
    Tampered       0.73      0.85      0.79      1025

    accuracy                           0.81      2524
   macro avg       0.81      0.82      0.81      2524
weighted avg       0.82      0.81      0.81      2524



In [7]:
model_to_check = effnet

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_to_check(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Authentic", "Tampered"]))

[[1180  319]
 [  47  978]]
              precision    recall  f1-score   support

   Authentic       0.96      0.79      0.87      1499
    Tampered       0.75      0.95      0.84      1025

    accuracy                           0.85      2524
   macro avg       0.86      0.87      0.85      2524
weighted avg       0.88      0.85      0.86      2524

